# Turkish Grammar Rules

This notebook is where you write and test your Turkish grammar rules.

**What you need to do:**
1. Run the first cell (Setup) — you do not change anything there
2. Fill in each rule cell — one rule per cell
3. Run the test cell at the bottom to check that your rules work

**Rules for writing a rule:**
- The class name and `name` field must match and have no spaces
- `check()` must always return a list — never `None`, even when there are no errors
- Every error message must start with `[HEURISTIC]`
- Use `sentence` for raw text checks (regex, character counting)
- Use `tokens` for word-level checks (list of lowercase words, no punctuation)

In [18]:
# ─────────────────────────────────────────────────────────────────────────────
# SETUP — run this cell first, do not change anything here
# ─────────────────────────────────────────────────────────────────────────────

from __future__ import annotations

import re
from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import List


@dataclass
class DetectedError:
    """
    Represents a single error found in a sentence.
    sentence   - the original sentence
    error_type - short label with no spaces
    message    - human-readable description, must start with [HEURISTIC]
    """
    sentence: str
    error_type: str
    message: str


class BaseRule(ABC):
    name: str = ""

    @abstractmethod
    def check(self, sentence: str, tokens: List[str]) -> List[DetectedError]:
        ...


def tokenize(sentence: str) -> List[str]:
    """Splits a Turkish sentence into lowercase words, removes punctuation."""
    return re.findall(r"[A-Za-z\u00C7\u00E7\u011E\u011F\u0130\u0131\u00D6\u00F6\u015E\u015F\u00DC\u00FC]{2,}", sentence.lower())


def test_rule(rule: BaseRule, sentence: str):
    """Runs a rule against one sentence and prints the result."""
    tokens = tokenize(sentence)
    errors = rule.check(sentence, tokens)
    print(f"Rule : {rule.name}")
    print(f"Input: {sentence}")
    if errors:
        for e in errors:
            print(f"  ✗  {e.message}")
    else:
        print("  ✓  No errors detected")
    print()


print("Setup complete.")

Setup complete.


---
## Example Rule


In [19]:
class SentenceCapitalizationRule(BaseRule):
    """
    [HEURISTIC] Flags sentences whose first alphabetic character is
    lowercase.

    Limitations:
      - Titles, headings, and list items often start with lowercase by
        convention and will be incorrectly flagged.
      - Sentences that begin with a digit (e.g. '3 dogs barked.') have
        their first *word* start with lowercase — the rule will flag 'd'
        in that example, which may or may not be the intended behaviour.
    """

    name = "SentenceCapitalization"

    def check(self, sentence: str, tokens: List[str]) -> List[DetectedError]:
        for ch in sentence.strip():
            if ch.isalpha():
                if ch.islower():
                    return [DetectedError(
                        sentence=sentence,
                        error_type="Capitalization",
                        message=f"[HEURISTIC] Sentence starts with lowercase '{ch}'.",
                    )]
                # First alpha char is uppercase — no error.
                break
        return []

print("=" * 60)
print("SentenceCapitalizationRule")
print("=" * 60)
test_rule(SentenceCapitalizationRule(), "this sentence starts with a lowercase letter.")   # should show ✗
test_rule(SentenceCapitalizationRule(), "This sentence starts with an uppercase letter.")  # should show ✓


SentenceCapitalizationRule
Rule : SentenceCapitalization
Input: this sentence starts with a lowercase letter.
  ✗  [HEURISTIC] Sentence starts with lowercase 't'.

Rule : SentenceCapitalization
Input: This sentence starts with an uppercase letter.
  ✓  No errors detected



---
## Rule 1

Write your first Turkish-specific rule below. Replace Rule1 with your actual rule name (no spaces). Describe what error the rule checks for in the docstring.

In [20]:
class AbbreviationApostropheRule(BaseRule):
    """
    [HEURISTIC] Flags common abbreviations that are missing
    an apostrophe before Turkish suffixes.

    Limitations:
      - Only checks abbreviations listed below.
      - Only checks the most common suffixes.
    """

    name = "AbbreviationApostrophe"

    _ABBREVIATIONS = {
        "tbmm", "tdk", "tüik", "sgk", "ösym",
        "yök", "tübitak", "trt", "thy", "abd", "ab"
    }

    _SUFFIXES = {
        "nin", "nın", "nun", "nün",
        "in", "ın", "un", "ün",
        "ye", "ya",
        "yi", "yı", "yu", "yü",
        "de", "da", "te", "ta",
        "den", "dan", "ten", "tan",
        "e", "a",
        "le", "la"
    }

    def check(self, sentence: str, tokens: List[str]) -> List[DetectedError]:

        errors: List[DetectedError] = []

        for token in tokens:

            # Eğer doğru kullanılmışsa geç
            if "'" in token or "’" in token:
                continue

            clean = token.lower().strip(".,!?;:()[]{}\"")

            for abbr in self._ABBREVIATIONS:

                if clean.startswith(abbr) and clean != abbr:

                    suffix = clean[len(abbr):]

                    if suffix in self._SUFFIXES:

                        errors.append(
                            DetectedError(
                                sentence=sentence,
                                error_type=self.name,
                                message=(
                                    f"[HEURISTIC] Abbreviation '{token}' "
                                    "should use an apostrophe before the suffix."
                                ),
                            )
                        )

                        break

        return errors


print("=" * 60)
print("AbbreviationApostropheRule")
print("=" * 60)

test_rule(AbbreviationApostropheRule(), "TBMMde yeni bir yasa kabul edildi.")      # ✗
test_rule(AbbreviationApostropheRule(), "TBMM'de yeni bir yasa kabul edildi.")     # ✓

AbbreviationApostropheRule
Rule : AbbreviationApostrophe
Input: TBMMde yeni bir yasa kabul edildi.
  ✗  [HEURISTIC] Abbreviation 'tbmmde' should use an apostrophe before the suffix.

Rule : AbbreviationApostrophe
Input: TBMM'de yeni bir yasa kabul edildi.
  ✓  No errors detected



---
## Rule 2

Write your second Turkish-specific rule below.

In [21]:
class QuestionParticleRule(BaseRule):
    """
    [HEURISTIC] Flags common cases where the Turkish question
    particle 'mi/mı/mu/mü' is attached to the previous word.

    Limitations:
      - Only detects the most common forms of the question particle.
      - May miss uncommon suffix combinations.
    """

    name = "QuestionParticle"

    _PATTERN = re.compile(
        r"\b\w+(mi|mı|mu|mü)(sun|sın|sin|sün|musun|mısın|misin|müsün)?\b",
        re.IGNORECASE,
    )

    def check(self, sentence: str, tokens: List[str]) -> List[DetectedError]:

        errors: List[DetectedError] = []

        for match in self._PATTERN.finditer(sentence):

            errors.append(
                DetectedError(
                    sentence=sentence,
                    error_type=self.name,
                    message=(
                        f"[HEURISTIC] '{match.group()}' may require a space "
                        f"before the question particle."
                    ),
                )
            )

        return errors


print("=" * 60)
print("QuestionParticleRule")
print("=" * 60)

test_rule(QuestionParticleRule(), "Geldinmi?")          # should show ✗
test_rule(QuestionParticleRule(), "Hazırmısın?")        # should show ✗
test_rule(QuestionParticleRule(), "Geldin mi?")         # should show ✓
test_rule(QuestionParticleRule(), "Hazır mısın?")       # should show ✓

QuestionParticleRule
Rule : QuestionParticle
Input: Geldinmi?
  ✗  [HEURISTIC] 'Geldinmi' may require a space before the question particle.

Rule : QuestionParticle
Input: Hazırmısın?
  ✗  [HEURISTIC] 'Hazırmısın' may require a space before the question particle.

Rule : QuestionParticle
Input: Geldin mi?
  ✓  No errors detected

Rule : QuestionParticle
Input: Hazır mısın?
  ✓  No errors detected



---
## Rule 3

Write your third Turkish-specific rule below.

In [22]:
class NumberSuffixRule(BaseRule):
    """
    [HEURISTIC] Flags numbers followed by a Turkish suffix
    without an apostrophe.

    Limitations:
      - Only detects numbers immediately followed by letters.
      - Does not verify whether the suffix itself is grammatically correct.
    """

    name = "NumberSuffix"

    _PATTERN = re.compile(
        r"\b\d+[a-zçğıöşü]+\b",
        re.IGNORECASE,
    )

    def check(self, sentence: str, tokens: List[str]) -> List[DetectedError]:

        errors: List[DetectedError] = []

        for match in self._PATTERN.finditer(sentence):

            errors.append(
                DetectedError(
                    sentence=sentence,
                    error_type=self.name,
                    message=(
                        f"[HEURISTIC] '{match.group()}' may require "
                        f"an apostrophe before the suffix."
                    ),
                )
            )

        return errors


print("=" * 60)
print("NumberSuffixRule")
print("=" * 60)

test_rule(NumberSuffixRule(), "Toplantı 2024te yapılacak.")      # should show ✗
test_rule(NumberSuffixRule(), "Saat 10da buluşalım.")            # should show ✗
test_rule(NumberSuffixRule(), "Toplantı 2024'te yapılacak.")     # should show ✓
test_rule(NumberSuffixRule(), "Saat 10'da buluşalım.")           # should show ✓

NumberSuffixRule
Rule : NumberSuffix
Input: Toplantı 2024te yapılacak.
  ✗  [HEURISTIC] '2024te' may require an apostrophe before the suffix.

Rule : NumberSuffix
Input: Saat 10da buluşalım.
  ✗  [HEURISTIC] '10da' may require an apostrophe before the suffix.

Rule : NumberSuffix
Input: Toplantı 2024'te yapılacak.
  ✓  No errors detected

Rule : NumberSuffix
Input: Saat 10'da buluşalım.
  ✓  No errors detected



---
## Rule 4

Write your third Turkish-specific rule below.

In [23]:
class BirçokRule(BaseRule):
    """
    [HEURISTIC] Flags the incorrect separate spelling
    of the Turkish word 'birçok'.

    Limitations:
      - Only detects the exact phrase 'bir çok'.
      - Does not check grammatical context.
    """

    name = "BirçokRule"

    _PATTERN = re.compile(
        r"\bbir çok\b",
        re.IGNORECASE,
    )

    def check(self, sentence: str, tokens: List[str]) -> List[DetectedError]:

        errors: List[DetectedError] = []

        for match in self._PATTERN.finditer(sentence):

            errors.append(
                DetectedError(
                    sentence=sentence,
                    error_type=self.name,
                    message=(
                        "[HEURISTIC] 'bir çok' should be written "
                        "as 'birçok'."
                    ),
                )
            )

        return errors


print("=" * 60)
print("BirçokRule")
print("=" * 60)

test_rule(BirçokRule(), "Bir çok insan geldi.")      # should show ✗
test_rule(BirçokRule(), "Bir çok kitap okudum.")     # should show ✗
test_rule(BirçokRule(), "Birçok insan geldi.")       # should show ✓
test_rule(BirçokRule(), "Birçok kitap okudum.")      # should show ✓

BirçokRule
Rule : BirçokRule
Input: Bir çok insan geldi.
  ✗  [HEURISTIC] 'bir çok' should be written as 'birçok'.

Rule : BirçokRule
Input: Bir çok kitap okudum.
  ✗  [HEURISTIC] 'bir çok' should be written as 'birçok'.

Rule : BirçokRule
Input: Birçok insan geldi.
  ✓  No errors detected

Rule : BirçokRule
Input: Birçok kitap okudum.
  ✓  No errors detected



---
## Rule 5

Write your third Turkish-specific rule below.

In [24]:
class HerŞeyRule(BaseRule):
    """
    [HEURISTIC] Flags the incorrect joined spelling
    of the Turkish phrase 'her şey'.

    Limitations:
      - Only detects the exact word 'herşey'.
      - Does not check grammatical context.
    """

    name = "HerŞeyRule"

    _PATTERN = re.compile(
        r"\bherşey\b",
        re.IGNORECASE,
    )

    def check(self, sentence: str, tokens: List[str]) -> List[DetectedError]:

        errors: List[DetectedError] = []

        for match in self._PATTERN.finditer(sentence):

            errors.append(
                DetectedError(
                    sentence=sentence,
                    error_type=self.name,
                    message=(
                        "[HEURISTIC] 'herşey' should be written "
                        "as 'her şey'."
                    ),
                )
            )

        return errors


print("=" * 60)
print("HerŞeyRule")
print("=" * 60)

test_rule(HerŞeyRule(), "Herşey yolunda.")          # should show ✗
test_rule(HerŞeyRule(), "Herşey hazır mı?")         # should show ✗
test_rule(HerŞeyRule(), "Her şey yolunda.")         # should show ✓
test_rule(HerŞeyRule(), "Her şey hazır mı?")        # should show ✓

HerŞeyRule
Rule : HerŞeyRule
Input: Herşey yolunda.
  ✗  [HEURISTIC] 'herşey' should be written as 'her şey'.

Rule : HerŞeyRule
Input: Herşey hazır mı?
  ✗  [HEURISTIC] 'herşey' should be written as 'her şey'.

Rule : HerŞeyRule
Input: Her şey yolunda.
  ✓  No errors detected

Rule : HerŞeyRule
Input: Her şey hazır mı?
  ✓  No errors detected



---
## Rule 6

Write your third Turkish-specific rule below

In [25]:
class RepeatedWordRule(BaseRule):
    """
    [HEURISTIC] Flags consecutive repeated words.

    Limitations:
      - Flags all consecutive repeated words.
      - May incorrectly flag valid Turkish reduplications
        such as 'yavaş yavaş' or 'teker teker'.
    """

    name = "RepeatedWord"

    def check(self, sentence: str, tokens: List[str]) -> List[DetectedError]:

        errors: List[DetectedError] = []

        for i in range(len(tokens) - 1):

            if tokens[i] == tokens[i + 1]:

                errors.append(
                    DetectedError(
                        sentence=sentence,
                        error_type=self.name,
                        message=(
                            f"[HEURISTIC] Repeated word detected: "
                            f"'{tokens[i]}'."
                        ),
                    )
                )

        return errors


print("=" * 60)
print("RepeatedWordRule")
print("=" * 60)

test_rule(RepeatedWordRule(), "Ben ben bunu yapmadım.")       # should show ✗
test_rule(RepeatedWordRule(), "Bu bu doğru değil.")           # should show ✗
test_rule(RepeatedWordRule(), "Ben bunu yapmadım.")           # should show ✓
test_rule(RepeatedWordRule(), "Bu doğru değil.")              # should show ✓

RepeatedWordRule
Rule : RepeatedWord
Input: Ben ben bunu yapmadım.
  ✗  [HEURISTIC] Repeated word detected: 'ben'.

Rule : RepeatedWord
Input: Bu bu doğru değil.
  ✗  [HEURISTIC] Repeated word detected: 'bu'.

Rule : RepeatedWord
Input: Ben bunu yapmadım.
  ✓  No errors detected

Rule : RepeatedWord
Input: Bu doğru değil.
  ✓  No errors detected



---
## Rule 7

Write your third Turkish-specific rule below

In [26]:
class DoubleSpaceRule(BaseRule):
    """
    [HEURISTIC] Flags double spaces within a sentence.
    """
    name = "DoubleSpaceRule"

    def check(self, sentence: str, tokens: List[str]) -> List[DetectedError]:
        if "  " in sentence:
            return [DetectedError(
                sentence=sentence,
                error_type=self.name,
                message="[HEURISTIC] Double space detected."
            )]
        return []

# Test
test_rule(DoubleSpaceRule(), "Bugün  hava çok güzel.") # ✗
test_rule(DoubleSpaceRule(), "Bugün hava çok güzel.")  # ✓

Rule : DoubleSpaceRule
Input: Bugün  hava çok güzel.
  ✗  [HEURISTIC] Double space detected.

Rule : DoubleSpaceRule
Input: Bugün hava çok güzel.
  ✓  No errors detected



---
## Rule 8

Write your third Turkish-specific rule below

In [27]:
class UnbalancedParenthesesRule(BaseRule):
    """
    [HEURISTIC] Flags sentences with unbalanced parentheses.
    """
    name = "UnbalancedParenthesesRule"

    def check(self, sentence: str, tokens: List[str]) -> List[DetectedError]:
        if sentence.count("(") != sentence.count(")"):
            return [DetectedError(
                sentence=sentence,
                error_type=self.name,
                message="[HEURISTIC] Unbalanced parentheses detected."
            )]
        return []

# Test
test_rule(UnbalancedParenthesesRule(), "Bu (cümle parantez hatası içeriyor.") # ✗
test_rule(UnbalancedParenthesesRule(), "Bu (cümle) parantez hatası içermiyor.")  # ✓

Rule : UnbalancedParenthesesRule
Input: Bu (cümle parantez hatası içeriyor.
  ✗  [HEURISTIC] Unbalanced parentheses detected.

Rule : UnbalancedParenthesesRule
Input: Bu (cümle) parantez hatası içermiyor.
  ✓  No errors detected



## Dataset

You need to write about 10 sentences per rule and about 40 correct sentences.

RULE 1:
- TBMMde yeni bir yasa kabul edildi.
- TDKnin sözlüğünü sık kullanıyorum.
- TRTye yeni bir program eklendi.
- YÖKten açıklama yapıldı.
- SGKya başvurumu yaptım.
- TÜBİTAKta yeni projeler destekleniyor.
- ABDde eğitim görmek istiyorum.
- ABye üyelik süreci devam ediyor.
- THYnin yeni seferleri başladı.
- TÜİKten enflasyon verileri açıklandı.


RULE 2:
- Geldinmi?
- Tamammı?
- Beni gördünmü?
- Hazırmısın?
- Mutlumusun?
- Gelecekmisin?
- Yaptınmı?
- Okudunmu?
- Duydunmu?
- Biliyormusun?


RULE 3:
-  Toplantı 2024te yapılacak.
-  15inci sırada oturuyorum.
-  Yarış 3üncü turda sona erdi.
-  5e kadar bekle.
-  Saat 10da buluşalım.
-  100den fazla kişi geldi.
-  7ye otobüs kalkacak.
-  12yi geçmeden dön.
-  29ekimde tören yapılacak.
-  8inci katta çalışıyorum.


RULE 4:
- Bir çok insan geldi.
- Bir çok öğrenci sınava girdi.
- Bir çok kitap okudum.
- Bir çok kişi katıldı.
- Bir çok ev satıldı.
- Bir çok fikir ortaya çıktı.
- Bir çok ülke katıldı.
- Bir çok çocuk oynuyordu.
- Bir çok soru çözdüm.
- Bir çok örnek bulduk.


RULE 5:
- Herşey yolunda.
- Herşey çok güzel olacak.
- Herşey hazır mı?
- Herşey bitti.
- Herşey istediğin gibi oldu.
- Herşey zamanla düzelir.
- Herşey kontrol altında.
- Herşey masanın üzerinde.
- Herşey doğru görünüyordu.
- Herşey senin için.


RULE 6:
- Ben ben bunu yapmadım.
- Bu bu doğru değil.
- Çok çok güzel olmuş.
- Sen sen söyledin.
- Bugün bugün yağmur yağacak.
- Hemenhemen değil, hemen hemen değil.
- Kitap kitap masada duruyor.
- Onlar onlar bizi bekliyordu.
- Ev ev dolaştılar.
-  Yavaşyavaş değil, yavaş yavaş yürü.

RULE 7:
- Bugün   hava çok güzel.  
- Sınav   zamanı yaklaşıyor.  
- Kitap  okumak çok  faydalıdır.  
- Bugün  ne  yapsak?  
- Yarın   pikniğe  gidiyoruz. 
- Bu  araba   çok  hızlı.  
- Bilgisayar    bozuldu  galiba. 
- Hava  bugün   çok  sıcak.  
- Ders  çalışmayı    çok  seviyorum.  
- İstanbul'da    trafik  çok  yoğun.  

RULE 8: 
- Yarınki toplantıya (Ahmet de katılacak.
- Bu raporu hazırlarken verileri )dikkatli( inceledik.
- Kitabı okumayı bitirdiğimde (çok etkilendim) diyecektim ama vazgeçtim.
- Okulun bahçesindeki ağaçlar )çok büyümüş(.
- Proje ödevini teslim etmek için (son gün yarın.
- Yarışmada birinci olmak için çok çalıştık (hedefimize ulaştık).
- Eğer istersen bugün dışarı çıkabiliriz )hava çok güzel(.
- Yemekten sonra bir kahve içelim (yorgunluğumuz geçsin.
- Dün akşamki konsere gitmeyi planlıyorduk )ancak biletler tükenmiş(.
- Konferans salonu (herkese yetecek kadar büyük değil.

Clean Dataset 

Rule 1: AbbreviationApostropheRule

- TBMM'de yeni bir yasa kabul edildi.  
- TDK'nin sözlüğünü sık kullanıyorum.  
- TRT'ye yeni bir program eklendi.  
- YÖK'ten açıklama yapıldı.  
- SGK'ya başvurumu yaptım. 


Rule 2: QuestionParticleRule

- Geldin mi?  
- Tamam mı?  
- Beni gördün mü?  
- Hazır mısın?  
- Mutlu musun?  

Rule 3: NumberSuffixRule

- Toplantı 2024'te yapılacak.  
- 15'inci sırada oturuyorum.  
- Yarış 3'üncü turda sona erdi.  
- 5'e kadar bekle.  
- Saat 10'da buluşalım.  


Rule 4 & 5: BirçokRule & HerŞeyRule

- Birçok insan toplantıya geldi.  
- Birçok öğrenci sınava başarılı girdi.  
- Birçok kitap okudum bu yaz.  
- Her şey yolunda görünüyor.  
- Her şey hazır mı diye sordum.  


Rule 6: RepeatedWordRule

- Ben bunu yapmadım.  
- Bu doğru değil.  
- Çok güzel olmuş.  
- Bugün yağmur yağacak.  
- Kitap masada duruyor.  

Rule 7: DoubleSpaceRule

- Yarınki toplantı için herkes hazırlıklarını tamamlasın.  
- Bu raporu incelediğimde verilerin çok tutarlı olduğunu gördüm.  
- Kitabı okumayı bitirdiğimde çok etkilendiğimi fark ettim.  
- Okulun bahçesindeki ağaçlar bu yıl çok güzel büyüdü.  
- Proje ödevini teslim etmek için yarın son günümüz.  


Rule 8: UnbalancedParenthesesRule

- Bu cümle hatalı değil (doğru kullanım).  
- Test hatalı değil (neyse ki).  
- Parantezleri kapamayı unutmadım (çok şükür).  
- İç içe parantez (hatası) düzeltildi.  
- Bu test (parantez) denemesi başarılı.  


- Hava bugün oldukça sıcak.  
- Yemekten sonra biraz dinlenmek için kahve içelim.  
- Dün akşamki konsere gitmeyi planlıyorduk.  
- Konferans salonu katılımcılar için genişti.  
- İstanbul'da trafik her zaman yoğundur.  
